# Explore taxonomy RDF — Virtuoso SPARQL queries

Queries the local Virtuoso instance via SPARQLWrapper. All queries run against named graphs, so cross-layer joins (taxonomy ↔ core ↔ properties ↔ NCBITaxon) work without limitations.

| Graph | URI |
|---|---|
| Core pathway RDF | `http://rdf-plantmetwiki.bioinformatics.nl/graph/pathways` |
| Taxonomy extra | `http://rdf-plantmetwiki.bioinformatics.nl/graph/gpml-taxonomy-extra` |
| Properties extra | `http://rdf-plantmetwiki.bioinformatics.nl/graph/gpml-properties-extra` |
| NCBITaxon ontology | `http://rdf-plantmetwiki.bioinformatics.nl/graph/ncbitaxon` |

**Kernel:** select `plantmetwiki-rdf` (register once with `python -m ipykernel install --user --name plantmetwiki-rdf`).

Once a query looks good, copy it to **[SPARQLQueries](https://github.com/pathway-lod/SPARQLQueries)**.

In [37]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

# ── Endpoint ──────────────────────────────────────────────────────────────────
# Local Virtuoso (recommended — fast, full data, all named graphs available)
ENDPOINT = "http://localhost:8890/sparql"
# Public endpoint (no local setup, but may be slower for large aggregations):
# ENDPOINT = "https://sparql-plantmetwiki.bioinformatics.nl/sparql"

# ── Named graph URIs ──────────────────────────────────────────────────────────
BASE      = "http://rdf-plantmetwiki.bioinformatics.nl"
G_CORE    = f"{BASE}/graph/pathways"
G_TAX     = f"{BASE}/graph/gpml-taxonomy-extra"
G_PROP    = f"{BASE}/graph/gpml-properties-extra"
G_NCBI    = f"{BASE}/graph/ncbitaxon"

# ── Common prefixes ───────────────────────────────────────────────────────────
PREFIXES = """
PREFIX wp:      <http://vocabularies.wikipathways.org/wp#>
PREFIX ncbi:    <http://purl.obolibrary.org/obo/NCBITaxon_>
PREFIX pmw:     <http://rdf-plantmetwiki.bioinformatics.nl/vocab/>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX rdf:     <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX dc:      <http://purl.org/dc/elements/1.1/>
"""

# ── Helper ────────────────────────────────────────────────────────────────────
def run_query(query: str) -> pd.DataFrame:
    """Run a SPARQL SELECT against the endpoint and return a DataFrame."""
    sp = SPARQLWrapper(ENDPOINT)
    sp.setQuery(PREFIXES + query)
    sp.setReturnFormat(JSON)
    results = sp.query().convert()
    rows = results["results"]["bindings"]
    if not rows:
        return pd.DataFrame()
    cols = list(rows[0].keys())
    return pd.DataFrame(
        [{c: r[c]["value"] for c in cols if c in r} for r in rows],
        columns=cols,
    )

print(f"Endpoint: {ENDPOINT}")
print("Ready.")

Endpoint: http://localhost:8890/sparql
Ready.


---
## 1. Pathway IRIs and PlantCyc IDs

Each pathway has a stable IRI (`/pathways/PC{n}_r{version}`) and a PlantCyc ID stored as `pmw:plantcycId`.

In [38]:
# Sample pathway IRI → PlantCyc ID
run_query(f"""
SELECT ?pathway_iri ?plantcyc_id
WHERE {{
  GRAPH <{G_PROP}> {{
    ?pathway_iri pmw:plantcycId ?plantcyc_id .
    FILTER(CONTAINS(STR(?pathway_iri), "/pathways/"))
  }}
}}
LIMIT 10
""")

,pathway_iri,plantcyc_id
0,http://rdf-plantmetwiki.bioinformatics.nl/path...,ARGININE-SYN4-PWY
1,http://rdf-plantmetwiki.bioinformatics.nl/path...,GLUCONEO-PWY
2,http://rdf-plantmetwiki.bioinformatics.nl/path...,GLUTAMATE-SYN2-PWY
3,http://rdf-plantmetwiki.bioinformatics.nl/path...,GLYSYN2-PWY
4,http://rdf-plantmetwiki.bioinformatics.nl/path...,ILEUSYN-PWY
5,http://rdf-plantmetwiki.bioinformatics.nl/path...,LEU-DEG2-PWY
6,http://rdf-plantmetwiki.bioinformatics.nl/path...,LEUSYN-PWY
7,http://rdf-plantmetwiki.bioinformatics.nl/path...,NONMEVIPP-PWY
8,http://rdf-plantmetwiki.bioinformatics.nl/path...,PANTO-PWY
9,http://rdf-plantmetwiki.bioinformatics.nl/path...,PWY-1001


In [39]:
# Total pathways — expected 1162
run_query(f"""
SELECT (COUNT(DISTINCT ?iri) AS ?pathways)
WHERE {{
  GRAPH <{G_PROP}> {{
    ?iri pmw:plantcycId ?id .
    FILTER(CONTAINS(STR(?iri), "/pathways/"))
  }}
}}
""")

,pathways
0,2478


---
## 2. GPML properties

All `<Property key="..." value="...">` elements are stored as blank nodes:
```turtle
?subject pmw:gpmlProperty [ pmw:key "..." ; pmw:value "..." ] .
```

In [40]:
# All unique property keys and their frequency — top 20
run_query(f"""
SELECT ?key (COUNT(?key) AS ?count)
WHERE {{
  GRAPH <{G_PROP}> {{
    ?subject pmw:gpmlProperty ?bn .
    ?bn pmw:key ?key .
  }}
}}
GROUP BY ?key
ORDER BY DESC(?count)
LIMIT 20
""")

,key,count
0,InstanceNameTemplate,54578
1,UniqueID,51528
2,Synonym_1,27711
3,Gibbs0,27049
4,Osmolarity,22564
5,Smiles,21976
6,NonStandardInchi,21563
7,MolecularWeight,20813
8,ChemicalFormula,20698
9,MonoisotopicMw,20695


In [41]:
# Original per-pathway species list from PlantCyc
# The pathway organism attribute is "Viridiplantae" in GPML, but the original
# multi-species string from PlantCyc is preserved as Property key="Organism".
run_query(f"""
SELECT ?pathway_iri ?original_species
WHERE {{
  GRAPH <{G_PROP}> {{
    ?pathway_iri pmw:gpmlProperty ?bn .
    ?bn pmw:key   "Organism" ;
        pmw:value ?original_species .
    FILTER(CONTAINS(STR(?pathway_iri), "/pathways/"))
  }}
}}
LIMIT 10
""")

,pathway_iri,original_species
0,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Brassica oleracea, Brassica juncea, Astragalus..."
1,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Chlamydomonas reinhardtii, Physcomitrium paten..."
2,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Populus trichocarpa, Arabidopsis thaliana"
3,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Lycopersicon hirsutum, Solanum, Solanum habroc..."
4,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Arabidopsis thaliana, Brassica napus, Limnanth..."
5,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Zea mays, Megathyrsus maximus, Urochloa panico..."
6,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Vitis vinifera, Zea mays, Cinnamomum tenuipile..."
7,http://rdf-plantmetwiki.bioinformatics.nl/path...,Arabidopsis thaliana
8,http://rdf-plantmetwiki.bioinformatics.nl/path...,Arabidopsis thaliana
9,http://rdf-plantmetwiki.bioinformatics.nl/path...,"Arabidopsis thaliana, Arabidopsis thaliana, Gl..."


---
## 3. Taxonomy overview

Basic counts and species distribution from the taxonomy-extra graph.

In [42]:
# Resources annotated with Viridiplantae — expected 2478 (1162 pathways + 1316 reactions)
run_query(f"""
SELECT (COUNT(DISTINCT ?resource) AS ?resources_with_viridiplantae)
WHERE {{
  GRAPH <{G_TAX}> {{
    ?resource wp:organism ncbi:33090 .
  }}
}}
""")

,resources_with_viridiplantae
0,2478


In [43]:
# Species distribution on DataNodes — top 20, with NCBI labels (cross-graph join)
run_query(f"""
SELECT ?taxon ?label (COUNT(DISTINCT ?node) AS ?node_count)
WHERE {{
  GRAPH <{G_TAX}> {{
    ?node wp:organism ?taxon .
    FILTER(?taxon != ncbi:33090)
    FILTER(CONTAINS(STR(?node), "/DataNode/"))
  }}
  OPTIONAL {{
    GRAPH <{G_NCBI}> {{ ?taxon rdfs:label ?label . }}
  }}
}}
GROUP BY ?taxon ?label
ORDER BY DESC(?node_count)
LIMIT 20
""")

,taxon,label,node_count
0,http://purl.obolibrary.org/obo/NCBITaxon_3702,Arabidopsis thaliana,7720
1,http://purl.obolibrary.org/obo/NCBITaxon_3847,Glycine max,1078
2,http://purl.obolibrary.org/obo/NCBITaxon_34305,Lotus japonicus,926
3,http://purl.obolibrary.org/obo/NCBITaxon_3055,Chlamydomonas reinhardtii,525
4,http://purl.obolibrary.org/obo/NCBITaxon_381124,Zea mays subsp. mays,513
5,http://purl.obolibrary.org/obo/NCBITaxon_4081,Solanum lycopersicum,422
6,http://purl.obolibrary.org/obo/NCBITaxon_4113,Solanum tuberosum,305
7,http://purl.obolibrary.org/obo/NCBITaxon_3888,Lathyrus oleraceus,236
8,http://purl.obolibrary.org/obo/NCBITaxon_46611,Abies grandis,230
9,http://purl.obolibrary.org/obo/NCBITaxon_39947,Oryza sativa Japonica Group,214


In [44]:
# Total species-annotated DataNodes (excluding Viridiplantae catch-all)
run_query(f"""
SELECT (COUNT(DISTINCT ?node) AS ?annotated_datanodes)
WHERE {{
  GRAPH <{G_TAX}> {{
    ?node wp:organism ?taxon .
    FILTER(?taxon != ncbi:33090)
    FILTER(CONTAINS(STR(?node), "/DataNode/"))
  }}
}}
""")

,annotated_datanodes
0,18762


---
## 4. Cross-layer queries (taxonomy ↔ core)

In [45]:
# Pathways containing at least one Arabidopsis thaliana DataNode
run_query(f"""
SELECT ?pathway ?title (COUNT(DISTINCT ?node) AS ?arabidopsis_nodes)
WHERE {{
  GRAPH <{G_CORE}> {{
    ?node wp:isPartOf ?pathway .
    ?pathway dc:title ?title .
  }}
  GRAPH <{G_TAX}> {{
    ?node wp:organism ncbi:3702 .
  }}
}}
GROUP BY ?pathway ?title
ORDER BY DESC(?arabidopsis_nodes)
LIMIT 20
""")

""


---
## 5. NCBITaxon coverage analysis

Checks which taxa present in the knowledge graph are absent from the loaded NCBITaxon ontology graph.

**Why taxa may be missing:**
- The OBO Foundry NCBITaxon release covers taxa relevant to biological ontologies — it is **not** a mirror of the full NCBI Taxonomy database.
- A taxon can also be missing because it was **deprecated and merged** into another taxon in a newer NCBI Taxonomy release.

**What is affected:**
Missing taxa can only affect GeneProduct and Protein nodes — these are the only node types that carry species-specific `wp:organism` annotations derived from PlantCyc's `proteins.dat` SPECIES field. Metabolites carry no direct species annotations.

In [46]:
# All unique taxa in the knowledge graph (excluding Viridiplantae top-level)
df_all_taxa = run_query(f"""
SELECT DISTINCT ?taxon
WHERE {{
  GRAPH <{G_TAX}> {{
    ?node wp:organism ?taxon .
    FILTER(?taxon != ncbi:33090)
  }}
}}
""")
print(f"Unique taxa in knowledge graph (excl. Viridiplantae): {len(df_all_taxa)}")

Unique taxa in knowledge graph (excl. Viridiplantae): 423


In [47]:
# Taxa present in the data but ABSENT from the NCBITaxon graph
# (no rdfs:label in graph/ncbitaxon means the taxon is not in the OBO Foundry release)
df_missing = run_query(f"""
SELECT ?taxon (COUNT(DISTINCT ?node) AS ?affected_nodes)
WHERE {{
  GRAPH <{G_TAX}> {{
    ?node wp:organism ?taxon .
    FILTER(?taxon != ncbi:33090)
  }}
  FILTER NOT EXISTS {{
    GRAPH <{G_NCBI}> {{ ?taxon rdfs:label ?label . }}
  }}
}}
GROUP BY ?taxon
ORDER BY DESC(?affected_nodes)
""")

# Extract NCBI ID for readability
df_missing["ncbi_id"] = df_missing["taxon"].str.extract(r'NCBITaxon_(\d+)')
print(f"Taxa absent from NCBITaxon graph: {len(df_missing)}")
df_missing[["ncbi_id", "taxon", "affected_nodes"]]

Taxa absent from NCBITaxon graph: 6


,ncbi_id,taxon,affected_nodes
0,101602,http://purl.obolibrary.org/obo/NCBITaxon_101602,17
1,48038,http://purl.obolibrary.org/obo/NCBITaxon_48038,14
2,121094,http://purl.obolibrary.org/obo/NCBITaxon_121094,11
3,283673,http://purl.obolibrary.org/obo/NCBITaxon_283673,6
4,135200,http://purl.obolibrary.org/obo/NCBITaxon_135200,4
5,23810,http://purl.obolibrary.org/obo/NCBITaxon_23810,2


In [48]:
# DataNodes affected by missing taxa — with node type and pathway title
df_affected = run_query(f"""
SELECT ?taxon ?node ?node_type ?pathway ?pathway_title
WHERE {{
  GRAPH <{G_TAX}> {{
    ?node wp:organism ?taxon .
    FILTER(?taxon != ncbi:33090)
  }}
  FILTER NOT EXISTS {{
    GRAPH <{G_NCBI}> {{ ?taxon rdfs:label ?label . }}
  }}
  GRAPH <{G_CORE}> {{
    ?node rdf:type ?node_type .
    ?node wp:isPartOf ?pathway .
    ?pathway dc:title ?pathway_title .
  }}
  FILTER(STRSTARTS(STR(?node_type), "http://vocabularies.wikipathways.org/wp#"))
}}
ORDER BY ?taxon ?pathway_title
""")

df_affected["ncbi_id"]   = df_affected["taxon"].str.extract(r'NCBITaxon_(\d+)')
df_affected["type_short"] = df_affected["node_type"].str.split("#").str[-1]
print(f"Affected DataNodes: {len(df_affected)}")
df_affected[["ncbi_id", "type_short", "pathway_title"]].value_counts(["ncbi_id", "type_short"])

KeyError: 'taxon'

In [ ]:
# Summary: impact as % of all species-annotated DataNodes
df_total = run_query(f"""
SELECT (COUNT(DISTINCT ?node) AS ?total_annotated)
WHERE {{
  GRAPH <{G_TAX}> {{
    ?node wp:organism ?taxon .
    FILTER(?taxon != ncbi:33090)
    FILTER(CONTAINS(STR(?node), "/DataNode/"))
  }}
}}
""")

total = int(df_total["total_annotated"].iloc[0])
missing_taxa   = len(df_missing)
affected_nodes = len(df_affected)
pct = affected_nodes / total * 100

print(f"Total species-annotated DataNodes : {total:,}")
print(f"Taxa absent from NCBITaxon graph  : {missing_taxa}")
print(f"Affected DataNodes                : {affected_nodes}  ({pct:.1f}% of annotated nodes)")
print()
print("Node types affected:")
print(df_affected["type_short"].value_counts().to_string())

In [ ]:
# Check whether any missing taxon has a replacement in NCBITaxon
# (deprecated taxa often have owl:deprecated + oboInOwl:replacedBy or rdfs:comment)
# Run one query per missing taxon IRI
missing_iris = df_missing["taxon"].tolist()

for taxon_iri in missing_iris:
    ncbi_id = taxon_iri.split("NCBITaxon_")[-1]
    df = run_query(f"""
    SELECT ?p ?o
    WHERE {{
      GRAPH <{G_NCBI}> {{
        <{taxon_iri}> ?p ?o .
      }}
    }}
    LIMIT 20
    """)
    if df.empty:
        print(f"NCBITaxon_{ncbi_id}: NOT in graph at all")
    else:
        print(f"NCBITaxon_{ncbi_id}: found {len(df)} triples")
        print(df.to_string(index=False))
    print()

---
## 6. Sandbox

Write new queries here. Use named graphs explicitly. When a query is validated, add it to **[SPARQLQueries](https://github.com/pathway-lod/SPARQLQueries)**.

In [ ]:
run_query(f"""
SELECT *
WHERE {{
  # ← write your query here
}}
LIMIT 20
""")